In [2]:
import geopandas as gpd
import pandas as pd

# Fetch live station metadata
URL = "https://gbfs.citibikenyc.com/gbfs/en/station_information.json"
df = pd.read_json(URL)["data"]["stations"]
df = pd.DataFrame(df)

# Save as CSV
df.to_csv("citibike_stations.csv", index=False)

# Convert to GeoJSON
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326"
)
gdf.to_file("citibike_stations.geojson", driver="GeoJSON")

# Save as Parquet
df.to_parquet("citibike_stations.parquet")

In [ ]:
# # import osmnx as ox

# # # Define the region
# # place_name = "USA"

# # # Query tags for Starbucks
# # tags = {"amenity": "cafe", "brand": "Starbucks"}

# # # Download points of interest
# # starbucks_gdf = ox.features_from_place(place_name, tags=tags)

# # # Filter down to essential columns (Name, Geometry, Address where available)
# # starbucks_gdf = starbucks_gdf.reset_index()
# # starbucks_df = starbucks_gdf[["name", "geometry"]].copy()

# # # Extract Lat/Lon coordinates
# # starbucks_df["latitude"] = starbucks_gdf.geometry.centroid.y
# # starbucks_df["longitude"] = starbucks_gdf.geometry.centroid.x

# # # Export to CSV, GeoJSON, or Parquet
# # starbucks_df.to_csv("data/starbucks/usa_starbucks.csv", index=False)
# # starbucks_gdf.to_parquet("data/starbucks/usa_starbucks.parquet")
# # starbucks_gdf.to_file("data/starbucks/usa_starbucks.geojson", driver="GeoJSON")

# import pandas as pd
# import osmnx as ox

# # List of US States
# us_states = [
#     "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
#     "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
#     "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana",
#     "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
#     "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
#     "New Hampshire", "New Jersey", "New Mexico", "New York",
#     "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
#     "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
#     "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
#     "West Virginia", "Wisconsin", "Wyoming"
# ]

# # Query tags - using brand:wikidata handles minor tagging variations
# tags = {"brand:wikidata": "Q37158"}  # Wikidata ID for Starbucks

# state_gdfs = []

# for state in us_states:
#     try:
#         place = f"{state}, USA"
#         gdf = ox.features_from_place(place, tags=tags)
#         if not gdf.empty:
#             state_gdfs.append(gdf)
#         print(f"Fetched {len(gdf)} Starbucks locations for {state}")
#     except Exception as e:
#         print(f"Skipped {state}: {e}")

# # Combine all state GeoDataFrames and drop duplicates
# full_gdf = pd.concat(state_gdfs, ignore_index=True)
# full_gdf = full_gdf.drop_duplicates(subset=["osmid"])

# # Extract clean Lat/Lon centroids (works for both Points and Polygons)
# full_gdf["latitude"] = full_gdf.geometry.centroid.y
# full_gdf["longitude"] = full_gdf.geometry.centroid.x

# # Export Tabular CSV (drop complex geometry column)
# df_export = pd.DataFrame(full_gdf.drop(columns=["geometry"]))
# df_export[["name", "latitude", "longitude"]].to_csv("usa_starbucks.csv", index=False)

# # Export Spatial GeoJSON / Parquet
# full_gdf.to_file("usa_starbucks.geojson", driver="GeoJSON")
# full_gdf.to_parquet("usa_starbucks.parquet")

/Users/sra/files/projects/spatial_practice/.venv/lib/python3.11/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 60 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


Skipped Alabama: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by NewConnectionError("HTTPSConnection(host='overpass-api.de', port=443): Failed to establish a new connection: [Errno 61] Connection refused"))


/Users/sra/files/projects/spatial_practice/.venv/lib/python3.11/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 1,886 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


KeyboardInterrupt: 

In [6]:
import geopandas as gpd

# Load national US County boundaries directly from Census servers
url = "https://www2.census.gov/geo/tiger/TIGER2025/COUNTY/tl_2025_us_county.zip"
counties_gdf = gpd.read_file(url)

# Save to your target format
counties_gdf.to_parquet("data/us_counties.parquet")

In [ ]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

# Query a valid Overture release path
query = """
SELECT 
    id,
    names.primary AS name,
    addresses[1].freeform AS address,
    addresses[1].locality AS city,
    addresses[1].region AS state,
    addresses[1].postcode AS zip,
    ST_Y(geometry) AS latitude,
    ST_X(geometry) AS longitude
FROM read_parquet('s3://overturemaps-us-west-2/release/2026-08-19.0/theme=places/type=place/*', filename=true, hive_partitioning=1)
WHERE addresses[1].country = 'US'
  AND (
      LOWER(names.primary) = 'starbucks'
      OR LOWER(brand.names.primary) = 'starbucks'
  );
"""

df_starbucks = con.execute(query).df()
df_starbucks.to_parquet("data/usa_starbucks.parquet")
df_starbucks.to_csv("data/usa_starbucks.csv", index=False)

print(f"Fetched {len(df_starbucks)} Starbucks locations in the US.")

IOException: IO Error: No files found that match the pattern "s3://overturemaps-us-west-2/release/2026-02-18.0/theme=places/type=place/*"

LINE 11: FROM read_parquet('s3://overturemaps-us-west-2/release/2026-02...
              ^